## **Setup**

In [ ]:
# Notebook autoreload
%load_ext autoreload
%autoreload 2

In [ ]:
# Path setup
from pathlib import Path
import sys

PROJECT_ROOT = Path().resolve().parents[0]  # notebooks/ is one level down
sys.path.append(str(PROJECT_ROOT / "src"))

In [ ]:
# Libraries
import os as os
import json as json
import torch
import pandas as pd

# Local
from chatGnT.config import CFG, ensure_dirs
from chatGnT.data import load, preprocess, tokenize, dataloaders
from chatGnT.models import train

# Setup
ensure_dirs(CFG)

## **Load Tokens & Make Vocab**

In [ ]:
# Read in tokens.json
with open(CFG.outputs_dir / "models/tokens_mt.json", "r") as f:
    tokens_mt = json.load(f)
with open(CFG.outputs_dir / "models/tokens_st.json", "r") as f:
    tokens_st = json.load(f)


In [ ]:
vocab_amt, vocab_ingred = tokenize.make_vocab_mt(tokens_mt)
inv_vocab_amt, inv_vocab_ingred = tokenize.invert_vocab_mt(vocab_amt, vocab_ingred)
tokens_padded_mt = tokenize.encode_tokens_mt(tokens_mt, vocab_amt, vocab_ingred)

# Save vocabs to file
with open(CFG.outputs_dir / "models/vocab_mt_amt.json", "w") as f:
    json.dump(vocab_amt, f)
with open(CFG.outputs_dir / "models/vocab_mt_ingred.json", "w") as f:
    json.dump(vocab_ingred, f)


vocab = tokenize.make_vocab_st(tokens_st)
inv_vocab = tokenize.invert_vocab_st(vocab)
tokens_padded_st = tokenize.encode_tokens_st(tokens_st, vocab)

# Save vocab to file
with open(CFG.outputs_dir / "models/vocab_st.json", "w") as f:
    json.dump(vocab, f)


In [ ]:
print("MT amount vocab ntokens:", len(vocab_amt))
print("First 6:", list(vocab_amt.items())[:6])
print("Last 3:", list(vocab_amt.items())[-3:])

print("MT ingredient vocab ntokens:", len(vocab_ingred))
print("First 6:", list(vocab_ingred.items())[:6])
print("Last 3:", list(vocab_ingred.items())[-3:])

print("ST vocab ntokens:", len(vocab))
print("First 6:", list(vocab.items())[:6])
print("Last 3:", list(vocab.items())[-3:])

In [ ]:
# Unzip the padded tokens
amt_seqs, ingred_seqs = zip(*tokens_padded_mt)
seqs = tokens_padded_st

# Convert to tensors
amt_tensor = torch.tensor(amt_seqs, dtype=torch.long)
ingred_tensor = torch.tensor(ingred_seqs, dtype=torch.long)
seqs_tensor = torch.tensor(seqs, dtype=torch.long)

# Check shape: (num_recipes, seq_len)
print(amt_tensor.shape, ingred_tensor.shape, seqs_tensor.shape)

## **Shared Config**

In [ ]:
config = {
    # model architecture
    "nhead": 2,

    # step schedular settings
    "scheduler_gamma": 0.95,
    "scheduler_step_size": 1,
    # reduce on plateau scheduler settings
    "scheduler_factor": 0.5,
    "scheduler_patience": 4,

    # dataloaders
    "split": 0.85,
    "seed": 42,

    # epochs, early stopping, & logs
    "log_interval": 6
}

In [ ]:
search_space = {
    "batch_size": [4, 8, 16],
    "learning_rate": [1e-3, 5e-4, 1e-4],
    "weight_decay": [1e-4, 1e-3, 1e-2],
    "scheduler_type": ["reduce_on_plateau", "step"],
    "ninp": [16, 32, 64],
    "nhid": [32, 64, 128],
    "nlayers": [1, 2]
}

## **Run Multi-Task Hyperparameter Search**

In [ ]:
search_base_config = {
    **config,
    "epochs": 200,
    "early_stop": 25,
    "ntoken_amt": len(vocab_amt),
    "ntoken_ingred": len(vocab_ingred),
    "model_version": "multi_task",
    "pad_id_amt": vocab_amt["<pad>"],
    "pad_id_ingred": vocab_ingred["<pad>"],
}

search_tensors = {
    "amt_tensor": amt_tensor,
    "ingred_tensor": ingred_tensor,
}

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
best_search_result, search_results = train.run_hyperparameter_search(
    base_config=search_base_config,
    search_space=search_space,
    tensors=search_tensors,
    device=device,
)

print("Best config:")
print(best_search_result["config"])
print(f"Best val loss: {best_search_result['best_val_loss']:.4f}")

# time = ~320 minutes

In [ ]:
search_summary = pd.DataFrame([
    {
        "trial": result["trial"],
        **{key: result["config"][key] for key in search_space},
        "best_val_loss": result["best_val_loss"],
    }
    for result in search_results
]).sort_values("best_val_loss").reset_index(drop=True)

In [ ]:
# Save search summary
search_summary.columns = search_summary.columns.str.replace("_", "-")
search_summary = search_summary.replace("_", "-", regex=True)
search_summary.to_csv(CFG.outputs_dir / "models/search_result_mt.csv", index=False)


## **Run Single-Task Hyperparameter Search**

In [ ]:
search_base_config = {
    **config,
    "epochs": 200,
    "early_stop": 25,
    "ntoken": len(vocab),
    "model_version": "single_task",
    "pad_id": vocab["<pad>"],
    "vocab": vocab,
}

search_tensors = {
    "tensor": seqs_tensor,
}

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
best_search_result, search_results = train.run_hyperparameter_search(
    base_config=search_base_config,
    search_space=search_space,
    tensors=search_tensors,
    device=device,
)

print("Best config:")
print(best_search_result["config"])
print(f"Best val loss: {best_search_result['best_val_loss']:.4f}")

# time = ~700 minutes

In [ ]:
search_summary = pd.DataFrame([
    {
        "trial": result["trial"],
        **{key: result["config"][key] for key in search_space},
        "best_val_loss": result["best_val_loss"],
    }
    for result in search_results
]).sort_values("best_val_loss").reset_index(drop=True)

In [ ]:
# Save search summary
search_summary.columns = search_summary.columns.str.replace("_", "-")
search_summary = search_summary.replace("_", "-", regex=True)
search_summary.to_csv(CFG.outputs_dir / "models/search_result_st.csv", index=False)
